In [1]:
import os, glob, shutil
from PIL import Image

ROOT = '/kaggle/working/levels'
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)

def prep(level, files, n=10):
    d = f'{ROOT}/L{level}'
    os.makedirs(d, exist_ok=True)
    saved = 0
    for f in files:
        if saved >= n:
            break
        try:
            im = Image.open(f).convert('RGB')
        except Exception:
            continue
        w, h = im.size
        if min(w, h) < 256:
            continue
        s = min(w, h)
        im = im.crop(((w-s)//2, (h-s)//2, (w+s)//2, (h+s)//2))
        im = im.resize((256, 256), Image.LANCZOS)
        im.save(f'{d}/{saved:05d}.png')
        saved += 1
    return saved

def spaced(lst, n=10):
    """n evenly spaced items, so one species or one painter doesn't dominate"""
    if len(lst) <= n:
        return lst
    step = len(lst) // n
    return lst[::step][:n]

L0 = sorted(glob.glob('/kaggle/input/datasets/denislukovnikov/ffhq256-images-only/ffhq256/*.png'))[:10]

L1 = sorted(glob.glob('/kaggle/input/datasets/badasstechie/celebahq-resized-256x256/celeba_hq_256/*.jpg'))[:10]

L2 = spaced(sorted(glob.glob('/kaggle/input/datasets/andrewmvd/animal-faces/afhq/val/**/*.jpg',
                             recursive=True)))

L3 = spaced(sorted(glob.glob('/kaggle/input/datasets/deewakarchakraborty/portrait-paintings/Images/*.jpg')))

# objects: original nine minus the park, plus the two new ones
old = sorted(glob.glob('/kaggle/input/datasets/aratisutar/nonface/*'))
old = [f for f in old if 'n3.' not in f]          # <-- park, adjust if it is a different file
new = sorted(glob.glob('/kaggle/input/datasets/aratisutar/nonface2/*'))
L4 = old + new

L5 = spaced(sorted(glob.glob('/kaggle/input/datasets/arnaud58/landscape-pictures/*.jpg')))

for lvl, files in enumerate([L0, L1, L2, L3, L4, L5]):
    got = prep(lvl, files)
    print('L' + str(lvl) + ':', got, 'images   (candidates:', len(files), ')')

L0: 10 images   (candidates: 10 )
L1: 10 images   (candidates: 10 )
L2: 10 images   (candidates: 10 )
L3: 10 images   (candidates: 10 )
L4: 10 images   (candidates: 10 )
L5: 10 images   (candidates: 10 )


In [2]:
import os, glob, shutil

# DPS code and weights
os.makedirs('/kaggle/working/dps', exist_ok=True)
!cp -r /kaggle/input/datasets/aratisutar/dps-code/diffusion-posterior-sampling-main/. /kaggle/working/dps/
os.makedirs('/kaggle/working/dps/models', exist_ok=True)
!cp /kaggle/input/datasets/aratisutar/dps-weights/ffhq_10m.pt /kaggle/working/dps/models/

# motionblur stub
os.makedirs('/kaggle/working/dps/motionblur', exist_ok=True)
open('/kaggle/working/dps/motionblur/__init__.py','w').close()
open('/kaggle/working/dps/motionblur/motionblur.py','w').write(
    'class Kernel:\n    def __init__(self,*a,**k):\n        raise NotImplementedError("stub")\n')

# move the level folders inside dps/data so the configs can use relative paths
for lvl in range(6):
    src = f'/kaggle/working/levels/L{lvl}'
    dst = f'/kaggle/working/dps/data/L{lvl}'
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f'{src}/*.png'):
        shutil.copy(f, dst)

# one config per level, everything else identical to the preliminary work
os.chdir('/kaggle/working/dps')
src = open('configs/super_resolution_config.yaml').read()
for lvl in range(6):
    open(f'configs/L{lvl}.yaml','w').write(
        src.replace('root: ./data/samples/', f'root: ./data/L{lvl}/'))

print(open('configs/L3.yaml').read())
for lvl in range(6):
    print(f'L{lvl}:', len(os.listdir(f'/kaggle/working/dps/data/L{lvl}')), 'images')
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'OFF')

conditioning:
  method: ps
  params:
    scale: 0.3

data:
  name: ffhq
  root: ./data/L3/

measurement:
  operator:
    name: super_resolution
    in_shape: !!python/tuple [1, 3, 256, 256]
    scale_factor: 4
  
  noise: 
    name: gaussian
    sigma: 0.05

L0: 10 images
L1: 10 images
L2: 10 images
L3: 10 images
L4: 10 images
L5: 10 images
GPU: Tesla T4


In [3]:
import subprocess, shutil, os, time

LEVELS = [0, 1, 2]     # run 1: [0,1,2]   run 2: [3,4,5]
os.chdir('/kaggle/working/dps')
start = time.time()

for lvl in LEVELS:
    imgs = [f'{i:05d}' for i in range(10)]
    for img in imgs:
        os.makedirs(f'/kaggle/working/P_L{lvl}_{img}', exist_ok=True)
    for rep in range(6):
        if all(os.path.exists(f'/kaggle/working/P_L{lvl}_{i}/{rep:02d}.png') for i in imgs):
            continue
        subprocess.run(['python','sample_condition.py',
            '--model_config=configs/model_config.yaml',
            '--diffusion_config=configs/diffusion_config.yaml',
            f'--task_config=configs/L{lvl}.yaml'], check=True, capture_output=True)
        for img in imgs:
            shutil.copy(f'results/super_resolution/recon/{img}.png',
                        f'/kaggle/working/P_L{lvl}_{img}/{rep:02d}.png')
            shutil.copy(f'results/super_resolution/label/{img}.png',
                        f'/kaggle/working/truth_L{lvl}_{img}.png')
        print(f'L{lvl} rep {rep}   ({(time.time()-start)/60:.1f} min)')
print('RUN DONE')

L0 rep 0   (37.6 min)
L0 rep 1   (75.4 min)
L0 rep 2   (113.1 min)
L0 rep 3   (150.8 min)
L0 rep 4   (188.5 min)
L0 rep 5   (226.3 min)
L1 rep 0   (264.0 min)
L1 rep 1   (301.7 min)
L1 rep 2   (339.4 min)
L1 rep 3   (377.2 min)
L1 rep 4   (414.9 min)
L1 rep 5   (452.6 min)
L2 rep 0   (490.3 min)
L2 rep 1   (528.0 min)
L2 rep 2   (565.7 min)
L2 rep 3   (603.4 min)
L2 rep 4   (641.1 min)
L2 rep 5   (678.8 min)
RUN DONE
